In [1]:
import ipywidgets as widgets

widgets.IntSlider(value=50, min=0, max=100, description='Value:')

IntSlider(value=50, description='Value:')

In [1]:
import ipywidgets as widgets

widgets.IntSlider(value=50, min=0, max=100, description='Value:')

IntSlider(value=50, description='Value:')

In [ ]:
import os
import rasterio

def clip_and_save_raster(src_path, dst_path):
    """
    Clips raster to largest dimensions divisible by 32 that fit within source image
    
    Args:
        src_path: Path to source TIFF file
        dst_path: Path to destination TIFF file
    """
    if os.path.exists(dst_path):
        return
        
    with rasterio.open(src_path) as src:
        data = src.read(1)
        profile = src.profile.copy()
        
        height, width = data.shape
        # Calculate largest dimensions divisible by 32
        new_height = (height // 32) * 32  # For 672 -> 672
        new_width = (width // 32) * 32    # For 576 -> 576
        

        start_y = (height - new_height) // 2
        start_x = (width - new_width) // 2
        
        clipped_data = data[start_y:start_y + new_height, 
                           start_x:start_x + new_width]
        
        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': rasterio.windows.transform(
                rasterio.windows.Window(start_x, start_y, new_width, new_height),
                src.transform
            )
        })

        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        
        with rasterio.open(dst_path, 'w', **profile) as dst:
            dst.write(clipped_data, 1)

clip_and_save_raster('./Albedo.tif', './Albedo_clipped.tif')

In [ ]:
from datetime import datetime
import shutil
def save_prediction_and_truth(model, test_loader, test_file_list, device='cuda'):
    """
    Save prediction and ground truth from test loader as georeferenced TIFFs
    and calculate error metrics.
    
    Args:
        model: The trained UNet model
        test_loader: DataLoader containing test data
        test_file_list: List of file paths for test data
        device: Device to run model on (default='cuda')
        
    Returns:
        dict: Dictionary containing MAE and RMSE metrics
    """
    model = model.to(device)
    model.eval()
    metrics = {}

    with torch.no_grad():
        # Get one sample
        sample = next(iter(test_loader))
        inputs = sample['input'].to(device)
        targets = sample['target'].to(device)
        mask = sample['mask'].to(device)
        
        # Get model prediction
        outputs = model(inputs)
        
        # Move to CPU and convert to numpy
        mask_np = mask.cpu().numpy().squeeze()
        targets_np = targets.cpu().numpy().squeeze()
        predicted_np = outputs.cpu().numpy().squeeze()
        
        # Apply mask
        predicted_np[~mask_np] = np.nan
        targets_np[~mask_np] = np.nan
        
        # Get corresponding LST file path and save outputs
        lst_tif_path = test_file_list[0]['LST.tif']
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        with rasterio.open(lst_tif_path) as src:
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)
            
            # Create output directory
            output_dir = "./Data/predictions"
            os.makedirs(output_dir, exist_ok=True)
            
            # Save prediction
            pred_filename = os.path.join(output_dir, f'predicted_LST_{timestamp}.tif')
            with rasterio.open(pred_filename, "w", **profile) as dst:
                dst.write(predicted_np.astype(np.float32), 1)
            
            # Save ground truth
            truth_filename = os.path.join(output_dir, f'ground_truth_LST_{timestamp}.tif')
            with rasterio.open(truth_filename, "w", **profile) as dst:
                dst.write(targets_np.astype(np.float32), 1)
            
            # Copy original LST file
            orig_filename = os.path.join(output_dir, f'original_LST_{timestamp}.tif')
            shutil.copy2(lst_tif_path.replace('Data/preprocess/', 'Data/'), orig_filename)
            
            # Calculate metrics for valid pixels
            valid_mask = ~np.isnan(predicted_np)
            if valid_mask.any():
                mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                metrics = {'mae': mae, 'rmse': rmse}
                
                print(f"Mean Absolute Error: {mae:.2f}°F")
                print(f"Root Mean Square Error: {rmse:.2f}°F")
            
            print(f"\nSaved files in {output_dir}/:")
            print(f"Predictions: {os.path.basename(pred_filename)}")
            print(f"Ground Truth: {os.path.basename(truth_filename)}")
            print(f"Original LST: {os.path.basename(orig_filename)}")
    
    return metrics

def calculate_rmse(model, test_loader, device='cuda'):
    model.eval()
    
    with torch.no_grad():
        sample = next(iter(test_loader))
        inputs = sample['input'].to(device)
        targets = sample['target'].to(device)
        mask = sample['mask'].to(device)
                
        outputs = model(inputs)
                
        mask_np = mask.cpu().numpy().squeeze()
        targets_np = targets.cpu().numpy().squeeze()
        predicted_np = outputs.cpu().numpy().squeeze()
                
        predicted_np[~mask_np] = np.nan
        targets_np[~mask_np] = np.nan                
                
        valid_mask = ~np.isnan(predicted_np)
        if valid_mask.any():
            rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
            print(f"Root Mean Square Error: {rmse:.2f}°F")
            return rmse        
        return None

In [ ]:
# Load the trained model
# checkpoint = torch.load('best_model.pth')

# metrics = save_prediction_and_truth(
#     model=model,
#     test_loader=data_module.test_dataloader(),
#     test_file_list=data_module.test_files,
# )

calculate_rmse(model, data_module.test_dataloader())